In [3]:
# Cellule 0 — Imports et configuration
import json, os
import pandas as pd
import numpy as np
from collections import defaultdict

# Installer le client Groq si pas déjà fait
# (à lancer une fois dans un terminal : pip install groq)
try:
    from groq import Groq
    print("✅ Client Groq disponible")
except ImportError:
    print("⚠️ Client Groq non installé")
    print("   Lance dans Anaconda Prompt (env llmkger) : pip install groq")

# Chemins
TRANS_DIR = r'D:\Memoire\mooccubeX\translated'
KG_DIR = r'D:\Memoire\LLMKGER\recbole_data_large\llmkger'
RESULTS_REC = r'D:\Memoire\LLMKGER\reco_results_final'

print("✅ Config OK")

✅ Client Groq disponible
✅ Config OK


In [4]:
# Cellule 1 — Charger les noms lisibles
print("Chargement des noms lisibles...")

course_name = {}
with open(os.path.join(TRANS_DIR, 'course_translated.json'), 'r', encoding='utf-8') as f:
    for line in f:
        try:
            obj = json.loads(line)
            cid = obj.get('id', '')
            name = obj.get('name', '').strip()
            if cid: course_name[cid] = name
        except: continue
print(f"  Cours : {len(course_name):,} noms")

concept_name = {}
with open(os.path.join(TRANS_DIR, 'concept_translated.json'), 'r', encoding='utf-8') as f:
    for line in f:
        try:
            obj = json.loads(line)
            cid = obj.get('id', '')
            name = obj.get('name', '').strip()
            if cid: concept_name[cid] = name
        except: continue
print(f"  Concepts : {len(concept_name):,} noms")

teacher_name = {}
with open(os.path.join(TRANS_DIR, 'teacher_translated.json'), 'r', encoding='utf-8') as f:
    for line in f:
        try:
            obj = json.loads(line)
            tid = obj.get('id', '')
            name = obj.get('name', '').strip()
            if tid: teacher_name[tid] = name
        except: continue
print(f"  Enseignants : {len(teacher_name):,} noms")

print("\n=== Exemples ===")
print(f"Cours : {list(course_name.items())[:3]}")
print(f"Concepts : {list(concept_name.items())[:3]}")
print(f"Profs : {list(teacher_name.items())[:3]}")

Chargement des noms lisibles...
  Cours : 3,781 noms
  Concepts : 637,568 noms
  Enseignants : 17,018 noms

=== Exemples ===
Cours : [('C_584313', 'Introduction to "Zi Zhi Tong Jian"'), ('C_584329', 'Calculus - Limit theory and functions of one variable'), ('C_584381', 'photojournalism')]
Concepts : [('K_神经部_组织学与胚胎学', 'Neural'), ('K_促甲状腺激素细胞_组织学与胚胎学', 'Thyroid stimulating hormonal cells'), ('K_嗜色细胞_组织学与胚胎学', 'Thopicophile cells')]
Profs : [('T_1', '刘燕妮'), ('T_2', '陈怡'), ('T_3', '程钢')]


In [5]:
import json

# Regarder la structure complète du fichier teacher_translated
path = r'D:\Memoire\mooccubeX\translated\teacher_translated.json'
print("=== Structure de teacher_translated.json ===\n")
with open(path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 5: break
        obj = json.loads(line)
        print(json.dumps(obj, ensure_ascii=False, indent=2))
        print("---")

=== Structure de teacher_translated.json ===

{
  "id": "T_1",
  "name": "刘燕妮",
  "name_en": "Yanni Liu",
  "about": "Graduated from the Philosophy Department of Peking University and is a lecturer at the School of Marxism at Tsinghua University. Courses offered include: Marxist Philosophy; Principles of Marxism; Selected Readings of Confucian Classics; Mental Health and Spiritual Evolution; \"University\", \"The Doctrine of the Mean\" and Modern Psychology; The Way of \"University\"; Confucian Ways of Self-cultivation; Participated in the compilation of \"Marxist Philosophy\" textbooks.",
  "job_title": "lecturer",
  "org_name": "Tsinghua University",
  "about_zh": "北大哲学系毕业，清华大学马克思主义学院讲师。开设过的课程，马克思主义哲学；马克思主义原理；儒家经典选读；心理健康与精神进化；《大学》《中庸》与现代心理学；《大学》之道；儒家修身之道；参与《马克思主义哲学》教材编写。",
  "job_title_zh": "讲师",
  "org_name_zh": "清华大学"
}
---
{
  "id": "T_2",
  "name": "陈怡",
  "name_en": "Yi Chen",
  "about": "Born in Chongqing in 1945, he graduated from Huazhong Institute of Technology (now Huazhong

In [6]:
import json, os

TRANS_DIR = r'D:\Memoire\mooccubeX\translated'

# Recharger les profs avec name_en, job_title, org_name, about
teacher_info = {}
with open(os.path.join(TRANS_DIR, 'teacher_translated.json'), 'r', encoding='utf-8') as f:
    for line in f:
        try:
            obj = json.loads(line)
            tid = obj.get('id', '')
            if tid:
                teacher_info[tid] = {
                    'name': obj.get('name_en') or obj.get('name', 'the instructor'),
                    'title': obj.get('job_title', ''),
                    'org': obj.get('org_name', ''),
                    'about': (obj.get('about', '') or '')[:300]  # limiter la bio
                }
        except: continue

print(f"Enseignants chargés : {len(teacher_info):,}")
print("\n=== Exemples ===")
for tid in ['T_1', 'T_2', 'T_3']:
    info = teacher_info.get(tid, {})
    print(f"{tid}: {info.get('name')} ({info.get('title')}, {info.get('org')})")

# Vérifier aussi si course_translated a des infos riches (about, etc.)
print("\n=== Structure de course_translated.json ===")
with open(os.path.join(TRANS_DIR, 'course_translated.json'), 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 2: break
        obj = json.loads(line)
        print(json.dumps(obj, ensure_ascii=False, indent=2)[:600])
        print("---")

Enseignants chargés : 17,018

=== Exemples ===
T_1: Yanni Liu (lecturer, Tsinghua University)
T_2: Yi Chen (professor, Tsinghua University)
T_3: Gang Cheng (Associate Professor, Tsinghua University)

=== Structure de course_translated.json ===
{
  "id": "C_584313",
  "name": "Introduction to \"Zi Zhi Tong Jian\"",
  "field": [
    "History",
    "Chinese Language and Literature"
  ],
  "prerequisites": "",
  "about": "Through the teacher's guidance, students can go deep into the interior of this classic text, gain an overview of thousands of years of history, improve their literacy in Chinese studies, and appreciate the wisdom of life.",
  "resource": [
    "{'titles': ['Lesson 1 Introduction and the Three Families' Division into Jin', 'Introduction', 'Introduction'], 'resource_id': 'V_849', 'chapter': '1.1.1'}",
    "{'titles': ['L
---
{
  "id": "C_584329",
  "name": "Calculus - Limit theory and functions of one variable",
  "field": [
    "Applied Economics",
    "Mathematics",
    "

In [7]:
# Recharger les cours avec toutes leurs infos
course_info = {}
with open(os.path.join(TRANS_DIR, 'course_translated.json'), 'r', encoding='utf-8') as f:
    for line in f:
        try:
            obj = json.loads(line)
            cid = obj.get('id', '')
            if cid:
                course_info[cid] = {
                    'name': obj.get('name', ''),
                    'field': obj.get('field', []),
                    'about': (obj.get('about', '') or '')[:400],
                }
        except: continue

print(f"Cours chargés : {len(course_info):,}")
print("\n=== Exemples ===")
for cid in ['C_584313', 'C_584329']:
    info = course_info.get(cid, {})
    print(f"{cid}: {info.get('name')}")
    print(f"   Domaines : {info.get('field')}")
    print(f"   Description : {info.get('about', '')[:150]}...")
    print()

print("✅ Métadonnées des cours prêtes")

Cours chargés : 3,781

=== Exemples ===
C_584313: Introduction to "Zi Zhi Tong Jian"
   Domaines : ['History', 'Chinese Language and Literature']
   Description : Through the teacher's guidance, students can go deep into the interior of this classic text, gain an overview of thousands of years of history, improv...

C_584329: Calculus - Limit theory and functions of one variable
   Domaines : ['Applied Economics', 'Mathematics', 'Physics', 'Theoretical Economics']
   Description : This course is a basic mathematics course in science and engineering. It provides a systematic and comprehensive introduction to the calculus of funct...

✅ Métadonnées des cours prêtes


In [8]:
# Structures d'extraction depuis le KG (si pas déjà en mémoire)
print("Construction des structures d'extraction...")

KG_DIR = r'D:\Memoire\LLMKGER\recbole_data_large\llmkger'
PREREQ_DIR = r'D:\Memoire\LLMKGER\prereq_enriched'

kg = pd.read_csv(os.path.join(KG_DIR, 'llmkger.kg'), sep='\t')
kg.columns = ['head','relation','tail']

# Concepts d'un cours
concepts_of_course = defaultdict(set)
for h,t in kg[kg['relation']=='belongs_to'][['head','tail']].itertuples(index=False):
    h,t=str(h),str(t)
    if h.startswith('K') and t.startswith('C_'): concepts_of_course[t].add(h)
    elif t.startswith('K') and h.startswith('C_'): concepts_of_course[h].add(t)

# Prérequis entre concepts (enrichis)
prereq_df = pd.read_csv(os.path.join(PREREQ_DIR, 'prerequisite_of_enriched.csv'), sep='\t')
prereqs_of_concept = defaultdict(set)
for h, r, t in prereq_df.itertuples(index=False):
    prereqs_of_concept[str(t)].add(str(h))

# Enseignant d'un cours
teacher_of_course = {}
for h,t in kg[kg['relation']=='taught_by'][['head','tail']].itertuples(index=False):
    h,t=str(h),str(t)
    if h.startswith('C_'): teacher_of_course[h] = str(t)
    elif t.startswith('C_'): teacher_of_course[t] = str(h)

print(f"  Cours avec concepts : {len(concepts_of_course):,}")
print(f"  Concepts avec prérequis : {len(prereqs_of_concept):,}")
print(f"  Cours avec enseignant : {len(teacher_of_course):,}")

# Fonction d'extraction des faits pour une paire (user, cours)
def extract_facts(user_courses, target_course):
    """
    user_courses : liste des cours déjà suivis par l'utilisateur (IDs C_...)
    target_course : cours recommandé (ID C_...)
    Retourne un dict de faits pour l'explication.
    """
    facts = {}
    # Infos du cours cible
    ci = course_info.get(target_course, {})
    facts['course_name'] = ci.get('name', target_course)
    facts['course_field'] = ci.get('field', [])
    facts['course_about'] = ci.get('about', '')

    # Concepts du cours cible
    target_concepts = concepts_of_course.get(target_course, set())
    facts['concepts'] = [concept_name.get(c, c) for c in list(target_concepts)[:8]]

    # Prérequis du cours cible (via ses concepts)
    target_prereqs = set()
    for c in target_concepts:
        target_prereqs |= prereqs_of_concept.get(c, set())

    # Concepts couverts par l'historique de l'utilisateur
    user_concepts = set()
    for uc in user_courses:
        user_concepts |= concepts_of_course.get(uc, set())

    covered_prereqs = target_prereqs & user_concepts
    missing_prereqs = target_prereqs - user_concepts
    facts['covered_prereqs'] = [concept_name.get(c, c) for c in list(covered_prereqs)[:6]]
    facts['missing_prereqs'] = [concept_name.get(c, c) for c in list(missing_prereqs)[:6]]

    # Enseignant
    tid = teacher_of_course.get(target_course)
    if tid and tid in teacher_info:
        facts['teacher'] = teacher_info[tid]
    else:
        facts['teacher'] = None

    # Cours suivis par l'utilisateur (noms)
    facts['user_history'] = [course_info.get(uc,{}).get('name', uc) for uc in user_courses[:5]]

    return facts

print("\n✅ Extracteur de faits prêt")

# Test sur un exemple fictif
test_user_courses = ['C_584329', 'C_584313']  # utilisateur a suivi calcul + histoire
test_target = 'C_584381'  # cours recommandé
sample_facts = extract_facts(test_user_courses, test_target)
print("\n=== Exemple de faits extraits ===")
for k, v in sample_facts.items():
    print(f"  {k}: {v}")

Construction des structures d'extraction...
  Cours avec concepts : 708
  Concepts avec prérequis : 19,910
  Cours avec enseignant : 2,796

✅ Extracteur de faits prêt

=== Exemple de faits extraits ===
  course_name: photojournalism
  course_field: ['Arts', 'Journalism and Communication']
  course_about: Master basic photography skills, understand how photo journalism works, train your observation and love for life, develop your aesthetic and critical skills for images, and develop a lifelong hobby.
  concepts: []
  covered_prereqs: []
  missing_prereqs: []
  teacher: None
  user_history: ['Calculus - Limit theory and functions of one variable', 'Introduction to "Zi Zhi Tong Jian"']


In [9]:
# Trouver des cours qui ONT des concepts (pour un meilleur exemple)
courses_with_concepts = [c for c in concepts_of_course.keys() if len(concepts_of_course[c]) >= 3]
print(f"Cours avec ≥3 concepts : {len(courses_with_concepts)}")

# Prendre un exemple riche
example_target = courses_with_concepts[0]
print(f"\nExemple de cours cible : {example_target}")
print(f"  Nom : {course_info.get(example_target,{}).get('name')}")
print(f"  Concepts : {[concept_name.get(c,c) for c in list(concepts_of_course[example_target])[:8]]}")

# Simuler un utilisateur qui a suivi 2-3 cours avec concepts
example_history = courses_with_concepts[1:4]
print(f"\nHistorique simulé : {[course_info.get(c,{}).get('name') for c in example_history]}")

# Extraire les faits
facts = extract_facts(example_history, example_target)
print("\n=== Faits extraits (exemple riche) ===")
for k, v in facts.items():
    if k == 'teacher' and v:
        print(f"  teacher: {v['name']} ({v['title']}, {v['org']})")
    else:
        print(f"  {k}: {v}")

Cours avec ≥3 concepts : 690

Exemple de cours cible : C_696679
  Nom : Circuit principle
  Concepts : ['Power side impedance', 'Steady-state expressions', 'Line current', 'Resonant conditions', 'Voltage ratio current', 'The value of the voltage', 'Inner impedance of power supply', 'Quadratic equations of feature roots']

Historique simulé : ['Living space and environmental design suitable for the elderly', 'Ideological and moral cultivation and legal basis', 'History of Chinese Architecture—Yuan, Ming and Qing Dynasties and Folk Residences']

=== Faits extraits (exemple riche) ===
  course_name: Circuit principle
  course_field: ['electrical engineering']
  course_about: Electricity is everywhere. The process of human civilization is almost the process of studying and utilizing electricity. Circuits are direct tools for using electricity to serve human beings. This course on Circuit Principles teaches you the basic methods of analyzing and designing circuits.
  concepts: ['Power side 

In [10]:
from groq import Groq

# ⚠️ Mets ta NOUVELLE clé Groq ici (celle que tu as régénérée)
GROQ_API_KEY = "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"
client = Groq(api_key=GROQ_API_KEY)

def build_prompt(facts, weights=None):
    """Construit le prompt pour le LLM à partir des faits."""
    p = f"""You are an educational recommendation assistant. Generate a clear, concise explanation (3-4 sentences) for why the following course is recommended to a learner.

COURSE RECOMMENDED: {facts['course_name']}
Field: {', '.join(facts['course_field']) if facts['course_field'] else 'N/A'}
Description: {facts['course_about']}

KEY CONCEPTS COVERED: {', '.join(facts['concepts'][:6]) if facts['concepts'] else 'N/A'}

LEARNER'S BACKGROUND (courses already taken):
{chr(10).join('- ' + h for h in facts['user_history']) if facts['user_history'] else '- New learner'}

PREREQUISITES:
- Already mastered: {', '.join(facts['covered_prereqs']) if facts['covered_prereqs'] else 'none identified'}
- Still needed: {', '.join(facts['missing_prereqs']) if facts['missing_prereqs'] else 'none'}
"""
    if facts.get('teacher'):
        t = facts['teacher']
        p += f"\nINSTRUCTOR: {t['name']}, {t['title']} at {t['org']}\n"

    if weights:
        p += f"\nRECOMMENDATION SIGNALS (how the system weighted this recommendation):\n"
        p += f"- Knowledge graph similarity: {weights['alpha']:.0%}\n"
        p += f"- Collaborative (similar learners): {weights['beta']:.0%}\n"
        p += f"- Prerequisite penalty: {weights['lambda']:.0%}\n"

    p += """\nWrite a natural, encouraging explanation that helps the learner understand why this course fits their journey. If prerequisites are missing, mention them gently as preparation. Do not invent facts not provided. Keep it under 80 words."""
    return p

# Test de génération
prompt = build_prompt(facts, weights={'alpha':0.35, 'beta':0.55, 'lambda':0.10})
print("=== PROMPT ===")
print(prompt)
print("\n" + "="*60)

# Appel Groq
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role":"user","content":prompt}],
    temperature=0.7,
    max_tokens=200,
)
explanation = response.choices[0].message.content
print("=== EXPLICATION GÉNÉRÉE ===")
print(explanation)

=== PROMPT ===
You are an educational recommendation assistant. Generate a clear, concise explanation (3-4 sentences) for why the following course is recommended to a learner.

COURSE RECOMMENDED: Circuit principle
Field: electrical engineering
Description: Electricity is everywhere. The process of human civilization is almost the process of studying and utilizing electricity. Circuits are direct tools for using electricity to serve human beings. This course on Circuit Principles teaches you the basic methods of analyzing and designing circuits.

KEY CONCEPTS COVERED: Power side impedance, Steady-state expressions, Line current, Resonant conditions, Voltage ratio current, The value of the voltage

LEARNER'S BACKGROUND (courses already taken):
- Living space and environmental design suitable for the elderly
- Ideological and moral cultivation and legal basis
- History of Chinese Architecture—Yuan, Ming and Qing Dynasties and Folk Residences

PREREQUISITES:
- Already mastered: none ident

In [11]:
def build_prompt_v2(facts, weights=None):
    p = f"""You are an educational advisor explaining a course recommendation to a learner. Write a personalized, specific explanation.

COURSE RECOMMENDED: {facts['course_name']} (Field: {', '.join(facts['course_field']) if facts['course_field'] else 'General'})
About: {facts['course_about']}
Key topics: {', '.join(facts['concepts'][:6]) if facts['concepts'] else 'N/A'}

LEARNER HAS TAKEN:
{chr(10).join('- ' + h for h in facts['user_history']) if facts['user_history'] else '- New learner (no history)'}

PREREQUISITE ANALYSIS:
- Already covered by learner: {', '.join(facts['covered_prereqs']) if facts['covered_prereqs'] else 'none'}
- Recommended preparation: {', '.join(facts['missing_prereqs'][:4]) if facts['missing_prereqs'] else 'none needed'}
"""
    if facts.get('teacher'):
        t = facts['teacher']
        p += f"Instructor: {t['name']}, {t['title']} at {t['org']}\n"
    if weights:
        dom = max(weights, key=lambda k: weights[k] if k!='lambda' else 0)
        dom_label = {'alpha':'knowledge-graph connections','beta':'learners with similar interests'}.get(dom,'multiple factors')
        p += f"\nThis recommendation is driven mainly by {dom_label}.\n"

    p += """
INSTRUCTIONS:
- Write 3-4 sentences, warm and specific.
- Reference the learner's actual background when relevant (note if this is a new direction for them).
- If prerequisites are missing, frame them as helpful preparation, not barriers.
- Mention the instructor if it adds value.
- Ground every claim in the facts above; invent nothing.
- Under 90 words."""
    return p

# Tester la v2
prompt_v2 = build_prompt_v2(facts, weights={'alpha':0.35,'beta':0.55,'lambda':0.10})
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role":"user","content":prompt_v2}],
    temperature=0.7, max_tokens=200,
)
print("=== EXPLICATION v2 ===")
print(response.choices[0].message.content)

=== EXPLICATION v2 ===
Based on learners with similar interests, I recommend Circuit Principles. This is a new direction for you, building on your diverse background. Associate Professor Xiucheng Liu will guide you. Preparation in loop current and line current will be helpful.


In [12]:
import numpy as np

# Charger les scores du modèle final (attention) pour identifier les top recommandations
# On a besoin de : score_attn (les scores), users_order, candidate_courses

# Vérifier ce qu'on a en mémoire
print("Vérification des données du modèle...")
for var in ['users_order', 'candidate_courses', 'id2item']:
    try:
        print(f"  ✅ {var} : {len(eval(var))}")
    except NameError:
        print(f"  ❌ {var} : absent (à recharger)")

# On a besoin de recharger le nécessaire pour faire des recommandations réelles
# Charger l'ordre des users et cours
RESULTS_REC = r'D:\Memoire\LLMKGER\reco_results_final'
users_order = pd.read_csv(os.path.join(RESULTS_REC,'S_emb_scores_final.csv'))['user_id'].astype(str).tolist()
print(f"\n  Users : {len(users_order):,}")
print(f"  Exemple user : {users_order[0]}")

Vérification des données du modèle...
  ❌ users_order : absent (à recharger)
  ❌ candidate_courses : absent (à recharger)
  ❌ id2item : absent (à recharger)

  Users : 53,897
  Exemple user : U_100090


In [14]:
from collections import defaultdict

# inter_feat est ici un DataFrame pandas — on accède aux colonnes différemment
inter_all = dataset.inter_feat

# Convertir en numpy selon le type
uid_field = config['USER_ID_FIELD']
iid_field = config['ITEM_ID_FIELD']

# Essayer les deux formats possibles
try:
    uu = inter_all[uid_field].to_numpy()
    ii = inter_all[iid_field].to_numpy()
except AttributeError:
    import numpy as np
    uu = np.array(inter_all[uid_field])
    ii = np.array(inter_all[iid_field])

user_history_courses = defaultdict(list)
for u, i in zip(uu, ii):
    u_ext = id2user.get(int(u))
    c_ext = id2item.get(int(i))
    if u_ext and c_ext:
        user_history_courses[u_ext].append(c_ext)

print(f"✅ Historiques : {len(user_history_courses):,} users")
print(f"   Exemple {users_order[0]} : {user_history_courses.get(users_order[0], [])[:3]}")
print(f"   Nb cours candidats : {len(candidate_courses):,}")

✅ Historiques : 53,897 users
   Exemple U_100090 : ['C_696954', 'C_676656', 'C_735046']
   Nb cours candidats : 2,812


In [15]:
import numpy as np, os
import pandas as pd

# Charger les cours test réels (ce que chaque user a suivi = bonnes recommandations)
df_test = pd.read_csv(os.path.join(RESULTS_REC, 'S_emb_scores_final.csv'))
# Colonnes : user_id, course_id (le cours test), S_emb

# Sélectionner un échantillon VARIÉ pour la démonstration
np.random.seed(42)

# On veut des exemples où le cours cible a des concepts (pour une explication riche)
sample_pairs = []
for _, row in df_test.iterrows():
    u = str(row['user_id'])
    c = str(row['course_id'])
    # garder si le cours a des concepts (explication riche possible)
    if len(concepts_of_course.get(c, set())) >= 2 and u in user_history_courses:
        sample_pairs.append((u, c))
    if len(sample_pairs) >= 100:
        break

print(f"Paires (user, cours recommandé) sélectionnées : {len(sample_pairs)}")
print("\n=== Exemples ===")
for u, c in sample_pairs[:5]:
    cname = course_info.get(c,{}).get('name', c)
    hist = [course_info.get(h,{}).get('name',h) for h in user_history_courses.get(u,[])[:2]]
    print(f"  {u} → {cname}")
    print(f"     (a suivi : {hist})")

Paires (user, cours recommandé) sélectionnées : 100

=== Exemples ===
  U_100090 → Start a new business
     (a suivi : ['Introduction to Business: 10 lessons to take you into the business world', 'Starting a business'])
  U_1000902 → software engineering
     (a suivi : ['software testing', 'software architecture'])
  U_1002389 → Basic Human Biology: Cells and Tissues
     (a suivi : ['Laser and its applications', 'Neural network theory and applications'])
  U_100281 → English Writing Guidance—Introduction to Writing
     (a suivi : ['dynamics', 'Medicine in the digital age'])
  U_100298 → Information literacy - a required course for academic research
     (a suivi : ['New Classroom in the E Era—Introduction to Online Education', 'Introduction to the basic principles of Marxism'])


In [16]:
import time, json

# Poids d'attention moyens (du modèle L2)
# Idéalement par paire, mais pour la démo on utilise une heuristique simple
def estimate_weights(facts):
    """Estime les poids selon les faits (approximation de l'attention)."""
    has_prereq_gap = len(facts['missing_prereqs']) > 3
    has_concepts = len(facts['concepts']) > 0
    # base
    a, b, l = 0.28, 0.52, 0.20
    if has_prereq_gap:
        l = 0.30; a = 0.25; b = 0.45  # plus de pénalité si prérequis manquants
    return {'alpha':a, 'beta':b, 'lambda':l}

# Générer les explications
explanations = []
print("Génération des explications...\n")

N_GENERATE = 100  # on génère les 100
for idx, (u, c) in enumerate(sample_pairs[:N_GENERATE]):
    facts = extract_facts(user_history_courses.get(u, []), c)
    weights = estimate_weights(facts)
    prompt = build_prompt_v2(facts, weights)
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role":"user","content":prompt}],
            temperature=0.7, max_tokens=200,
        )
        expl = response.choices[0].message.content.strip()
    except Exception as e:
        expl = f"[Erreur: {e}]"
        time.sleep(2)

    explanations.append({
        'user': u, 'course': c,
        'course_name': facts['course_name'],
        'user_history': facts['user_history'],
        'concepts': facts['concepts'][:5],
        'missing_prereqs': facts['missing_prereqs'][:4],
        'weights': weights,
        'explanation': expl
    })
    if (idx+1) % 10 == 0:
        print(f"  {idx+1}/{N_GENERATE} générées...")
    time.sleep(0.5)  # éviter le rate limit

print(f"\n✅ {len(explanations)} explications générées")

# Sauvegarder
out_dir = r'D:\Memoire\LLMKGER\explanations'
os.makedirs(out_dir, exist_ok=True)
with open(os.path.join(out_dir, 'explanations_llmkger.json'), 'w', encoding='utf-8') as f:
    json.dump(explanations, f, ensure_ascii=False, indent=2)
print(f"💾 Sauvegardé : {out_dir}\\explanations_llmkger.json")

Génération des explications...



15 Jul 21:55    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:55    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:55    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:55    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:55    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:55    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:55    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:55    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:55    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:55    INFO  HTTP Request: P

  10/100 générées...


15 Jul 21:55    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:55    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:55    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:56    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:56    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:56    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:56    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:56    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:56    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:56    INFO  HTTP Request: P

  20/100 générées...


15 Jul 21:56    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:56    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:56    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:56    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:56    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:56    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:56    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:56    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:56    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:56    INFO  HTTP Request: P

... (11 outputs left out)

15 Jul 21:58    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
15 Jul 21:58    INFO  Retrying request to /openai/v1/chat/completions in 2.000000 seconds
15 Jul 21:58    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:58    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
15 Jul 21:58    INFO  Retrying request to /openai/v1/chat/completions in 2.000000 seconds
15 Jul 21:59    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:59    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
15 Jul 21:59    INFO  Retrying request to /openai/v1/chat/completions in 2.000000 seconds
15 Jul 21:59    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:59    INFO  HTTP Request: POST ht

  90/100 générées...


15 Jul 21:59    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
15 Jul 21:59    INFO  Retrying request to /openai/v1/chat/completions in 2.000000 seconds
15 Jul 21:59    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:59    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
15 Jul 21:59    INFO  Retrying request to /openai/v1/chat/completions in 2.000000 seconds
15 Jul 21:59    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:59    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
15 Jul 21:59    INFO  Retrying request to /openai/v1/chat/completions in 2.000000 seconds
15 Jul 21:59    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 21:59    INFO  HTTP Request: POST ht

  100/100 générées...

✅ 100 explications générées
💾 Sauvegardé : D:\Memoire\LLMKGER\explanations\explanations_llmkger.json


In [17]:
import json, os

out_dir = r'D:\Memoire\LLMKGER\explanations'
with open(os.path.join(out_dir, 'explanations_llmkger.json'), 'r', encoding='utf-8') as f:
    explanations = json.load(f)

print(f"Total : {len(explanations)} explications\n")
print("="*75)

for e in explanations[:5]:
    print(f"\n👤 USER : {e['user']}")
    print(f"📚 A suivi : {', '.join(e['user_history'][:3])}")
    print(f"🎯 RECOMMANDÉ : {e['course_name']}")
    if e['concepts']:
        print(f"🔑 Concepts : {', '.join(e['concepts'][:4])}")
    if e['missing_prereqs']:
        print(f"⚠️  Prérequis à préparer : {', '.join(e['missing_prereqs'][:3])}")
    print(f"⚖️  Poids : α={e['weights']['alpha']:.2f} β={e['weights']['beta']:.2f} λ={e['weights']['lambda']:.2f}")
    print(f"\n💬 EXPLICATION :")
    print(f"   {e['explanation']}")
    print("\n" + "="*75)

Total : 100 explications


👤 USER : U_100090
📚 A suivi : Introduction to Business: 10 lessons to take you into the business world, Starting a business, Green Innovation Theory and Practice
🎯 RECOMMANDÉ : Start a new business
🔑 Concepts : Net assets, Project risk, Pseudo-demand, The present value of continuous value
⚠️  Prérequis à préparer : Changes in fair value, Financial assets, Profit manipulation
⚖️  Poids : α=0.25 β=0.45 λ=0.30

💬 EXPLICATION :
   Building on your foundation in business, I recommend "Starting a New Enterprise" to deepen your knowledge. Having taken courses like "Introduction to Business" and "Innovation and Entrepreneurship Practice", you're well-prepared. Professor Jian Gao will guide you through key topics like project risk and valuation.


👤 USER : U_1000902
📚 A suivi : software testing, software architecture, Database Systems (Medium): Modeling and Design
🎯 RECOMMANDÉ : software engineering
🔑 Concepts : System class, Control code, Enable, Large data volume
⚠️

In [18]:
def build_prompt_v3(facts, weights=None):
    """Prompt renforcé : strictement ancré dans les faits, pas d'extrapolation."""
    p = f"""You are an educational advisor. Explain why a course is recommended, using ONLY the facts provided below. Do NOT infer, assume, or add any information not explicitly listed.

=== FACTS (use only these) ===
COURSE RECOMMENDED: {facts['course_name']}
Field: {', '.join(facts['course_field']) if facts['course_field'] else 'General'}
Description: {facts['course_about']}
Key topics in this course: {', '.join(facts['concepts'][:6]) if facts['concepts'] else 'not specified'}

LEARNER'S COMPLETED COURSES (exact titles):
{chr(10).join('- ' + h for h in facts['user_history']) if facts['user_history'] else '- No prior courses (new learner)'}

PREREQUISITES:
- Covered by learner: {', '.join(facts['covered_prereqs']) if facts['covered_prereqs'] else 'none identified'}
- Recommended preparation: {', '.join(facts['missing_prereqs'][:4]) if facts['missing_prereqs'] else 'none needed'}
"""
    if facts.get('teacher'):
        t = facts['teacher']
        p += f"Instructor: {t['name']}, {t['title']} at {t['org']}\n"
    if weights:
        dom = 'similar learners' if weights['beta'] >= weights['alpha'] else 'knowledge-graph connections'
        p += f"Main recommendation driver: {dom}\n"

    p += """
=== STRICT RULES ===
1. Use ONLY the exact course titles and topics listed above. Do NOT rename, reinterpret, or generalize them (e.g., do not turn "Neural network theory" into "neuroscience").
2. Do NOT invent fields of study, skills, or connections not stated in the facts.
3. If the recommended course is unrelated to the learner's history, say so honestly (e.g., "this explores a new area").
4. Reference the instructor only with the exact name given.
5. Write 3-4 warm, natural sentences, under 85 words.

Write the explanation now:"""
    return p

# Tester la v3 sur le cas problématique (U_1002389 - neural network → biology)
test_pair = ('U_1002389', 'C_' + [c for c in sample_pairs if c[0]=='U_1002389'][0][1].split('_')[1] if any(p[0]=='U_1002389' for p in sample_pairs) else sample_pairs[2][1])
# Prendre directement le 3ème exemple
u, c = sample_pairs[2]
facts = extract_facts(user_history_courses.get(u, []), c)
weights = estimate_weights(facts)

prompt_v3 = build_prompt_v3(facts, weights)
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role":"user","content":prompt_v3}],
    temperature=0.5,  # température plus basse = plus fidèle aux faits
    max_tokens=200,
)
print("=== CAS TEST : U_1002389 (Neural network → Biology) ===")
print(f"Historique : {facts['user_history']}")
print(f"Recommandé : {facts['course_name']}")
print(f"\n💬 EXPLICATION v3 (renforcée) :")
print(f"   {response.choices[0].message.content.strip()}")

15 Jul 22:07    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


=== CAS TEST : U_1002389 (Neural network → Biology) ===
Historique : ['Laser and its applications', 'Neural network theory and applications', 'Flexible electronics technology', 'LCD technology', 'cognitive neuroscience']
Recommandé : Basic Human Biology: Cells and Tissues

💬 EXPLICATION v3 (renforcée) :
   I recommend Basic Human Biology: Cells and Tissues. This explores a new area. Femke Buisman-Pijlman teaches it. Similar learners took this course.


In [19]:
def build_prompt_v25(facts, weights=None):
    """Équilibre : riche comme v2, mais fidèle aux titres exacts."""
    p = f"""You are an educational advisor explaining a course recommendation to a learner. Write a warm, personalized, specific explanation.

COURSE RECOMMENDED: {facts['course_name']} (Field: {', '.join(facts['course_field']) if facts['course_field'] else 'General'})
About: {facts['course_about']}
Key topics: {', '.join(facts['concepts'][:6]) if facts['concepts'] else 'N/A'}

LEARNER HAS COMPLETED:
{chr(10).join('- ' + h for h in facts['user_history']) if facts['user_history'] else '- New learner (no history)'}

PREREQUISITE ANALYSIS:
- Already covered: {', '.join(facts['covered_prereqs']) if facts['covered_prereqs'] else 'none'}
- Recommended preparation: {', '.join(facts['missing_prereqs'][:4]) if facts['missing_prereqs'] else 'none needed'}
"""
    if facts.get('teacher'):
        t = facts['teacher']
        p += f"Instructor: {t['name']}, {t['title']} at {t['org']}\n"
    if weights:
        dom = 'learners with similar interests' if weights['beta'] >= weights['alpha'] else 'knowledge-graph connections'
        p += f"Main driver: {dom}\n"

    p += """
INSTRUCTIONS:
- Write 3-4 warm, natural sentences (under 85 words).
- Reference the learner's ACTUAL completed courses by their real titles when relevant.
- Use the exact course topic names — do not rename or generalize them.
- If this is a new direction for the learner, acknowledge it positively.
- Frame missing prerequisites as helpful preparation, not barriers.
- Mention the instructor when it adds value.
- Base everything on the facts above; do not fabricate.

Write the explanation:"""
    return p

# Tester sur le même cas
u, c = sample_pairs[2]
facts = extract_facts(user_history_courses.get(u, []), c)
weights = estimate_weights(facts)
prompt_v25 = build_prompt_v25(facts, weights)
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role":"user","content":prompt_v25}],
    temperature=0.6, max_tokens=200,
)
print("=== v2.5 (équilibrée) sur U_1002389 ===")
print(f"   {response.choices[0].message.content.strip()}")

15 Jul 22:08    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


=== v2.5 (équilibrée) sur U_1002389 ===
   I think you'll find Basic Human Biology: Cells and Tissues fascinating, building on concepts from cognitive neuroscience. This course introduces cells and tissues, a new direction for you. Femke Buisman-Pijlman will guide you through key topics like Neurons and Extracellular proteins. Prior knowledge from Laser and its applications and Neural network theory and applications will provide a solid foundation.


In [20]:
import time, json, os

explanations_v25 = []
print("Regénération avec le prompt v2.5...\n")

for idx, (u, c) in enumerate(sample_pairs[:100]):
    facts = extract_facts(user_history_courses.get(u, []), c)
    weights = estimate_weights(facts)
    prompt = build_prompt_v25(facts, weights)
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role":"user","content":prompt}],
            temperature=0.6, max_tokens=200,
        )
        expl = response.choices[0].message.content.strip()
    except Exception as e:
        expl = f"[Erreur: {e}]"
        time.sleep(2)
    explanations_v25.append({
        'user': u, 'course': c, 'course_name': facts['course_name'],
        'user_history': facts['user_history'], 'concepts': facts['concepts'][:5],
        'missing_prereqs': facts['missing_prereqs'][:4], 'weights': weights,
        'explanation': expl
    })
    if (idx+1) % 20 == 0: print(f"  {idx+1}/100...")
    time.sleep(0.5)

out_dir = r'D:\Memoire\LLMKGER\explanations'
with open(os.path.join(out_dir, 'explanations_llmkger.json'), 'w', encoding='utf-8') as f:
    json.dump(explanations_v25, f, ensure_ascii=False, indent=2)
print(f"\n✅ 100 explications v2.5 sauvegardées")

Regénération avec le prompt v2.5...



15 Jul 22:11    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:11    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:11    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:11    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:11    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:11    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:11    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:11    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:11    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:11    INFO  HTTP Request: P

  20/100...


15 Jul 22:11    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:11    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:11    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:11    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:11    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:11    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:11    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:11    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:11    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:12    INFO  HTTP Request: P

  40/100...


15 Jul 22:12    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
15 Jul 22:12    INFO  Retrying request to /openai/v1/chat/completions in 2.000000 seconds
15 Jul 22:12    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:12    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
15 Jul 22:12    INFO  Retrying request to /openai/v1/chat/completions in 2.000000 seconds
15 Jul 22:12    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:12    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
15 Jul 22:12    INFO  Retrying request to /openai/v1/chat/completions in 2.000000 seconds
15 Jul 22:12    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:12    INFO  HTTP Request: POST ht

  60/100...


15 Jul 22:13    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
15 Jul 22:13    INFO  Retrying request to /openai/v1/chat/completions in 2.000000 seconds
15 Jul 22:13    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:13    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
15 Jul 22:13    INFO  Retrying request to /openai/v1/chat/completions in 2.000000 seconds
15 Jul 22:13    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:13    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
15 Jul 22:13    INFO  Retrying request to /openai/v1/chat/completions in 2.000000 seconds
15 Jul 22:13    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:13    INFO  HTTP Request: POST ht

  80/100...


15 Jul 22:14    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
15 Jul 22:14    INFO  Retrying request to /openai/v1/chat/completions in 2.000000 seconds
15 Jul 22:14    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:14    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
15 Jul 22:14    INFO  Retrying request to /openai/v1/chat/completions in 2.000000 seconds
15 Jul 22:14    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:14    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
15 Jul 22:14    INFO  Retrying request to /openai/v1/chat/completions in 2.000000 seconds
15 Jul 22:14    INFO  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
15 Jul 22:14    INFO  HTTP Request: POST ht

  100/100...

✅ 100 explications v2.5 sauvegardées


In [21]:
import json, os

RESULTS_REC = r'D:\Memoire\LLMKGER\reco_results_final'

# Vérifier le contenu des résultats E-KGAT
ekgat_path = os.path.join(RESULTS_REC, 'EKGAT_results.json')
if os.path.exists(ekgat_path):
    with open(ekgat_path) as f:
        ekgat = json.load(f)
    print("=== Contenu EKGAT_results.json ===")
    print(json.dumps(ekgat, indent=2)[:800])

# E-KGAT génère typiquement des explications basées sur les chemins du KG
# (path-based explanations). Voyons si on a des chemins sauvegardés.
print("\n=== Fichiers liés à E-KGAT ===")
for f in os.listdir(RESULTS_REC):
    if 'ekgat' in f.lower() or 'EKGAT' in f:
        print(f"  {f}")

=== Contenu EKGAT_results.json ===
{
  "recall@5": 0.06432640035623505,
  "ndcg@5": 0.04054520374062259,
  "precision@5": 0.012865280071247008,
  "hit@5": 0.06432640035623505,
  "mrr@5": 0.03279650691751551,
  "recall@10": 0.10878156483663284,
  "ndcg@10": 0.05479068692894027,
  "precision@10": 0.010878156483663285,
  "hit@10": 0.10878156483663284,
  "mrr@10": 0.038600382976229496
}

=== Fichiers liés à E-KGAT ===
  EKGAT_model.pt
  EKGAT_results.json


In [1]:
import json, os

# Charger les explications LLM déjà générées
out_dir = r'D:\Memoire\LLMKGER\explanations'
with open(os.path.join(out_dir, 'explanations_llmkger.json'), 'r', encoding='utf-8') as f:
    explanations_llm = json.load(f)

print(f"Explications LLM chargées : {len(explanations_llm)}")

# Générer les explications TEMPLATE façon E-KGAT (basées sur les chemins KG)
def ekgat_template_explanation(facts):
    """Génère une explication template rigide, façon E-KGAT (path-based)."""
    course = facts['course_name']
    history = facts['user_history']
    concepts = facts['concepts']

    # Template basé sur les chemins du KG (comme E-KGAT)
    if history and concepts:
        return (f"Recommended: {course}. "
                f"Path: User → interacted with → {history[0]} → "
                f"shares concept → {concepts[0]} → concept of → {course}.")
    elif history:
        return (f"Recommended: {course}. "
                f"Path: User → interacted with → {history[0]} → "
                f"collaborative similarity → {course}.")
    else:
        return (f"Recommended: {course}. "
                f"Path: cold-start → popularity-based → {course}.")

# Générer pour toutes les paires
comparison = []
for e in explanations_llm:
    facts_min = {
        'course_name': e['course_name'],
        'user_history': e['user_history'],
        'concepts': e['concepts'],
    }
    ekgat_expl = ekgat_template_explanation(facts_min)
    comparison.append({
        'user': e['user'],
        'course_name': e['course_name'],
        'ekgat_explanation': ekgat_expl,
        'llmkger_explanation': e['explanation'],
    })

# Afficher 3 comparaisons côte à côte
print("\n" + "="*75)
for c in comparison[:3]:
    print(f"\n🎯 COURS : {c['course_name']}")
    print(f"\n📋 E-KGAT (template) :")
    print(f"   {c['ekgat_explanation']}")
    print(f"\n✨ LLMKGER (LLM) :")
    print(f"   {c['llmkger_explanation']}")
    print("\n" + "="*75)

# Sauvegarder
with open(os.path.join(out_dir, 'comparison_ekgat_vs_llmkger.json'), 'w', encoding='utf-8') as f:
    json.dump(comparison, f, ensure_ascii=False, indent=2)
print(f"\n💾 Comparaison sauvegardée")

Explications LLM chargées : 100


🎯 COURS : Start a new business

📋 E-KGAT (template) :
   Recommended: Start a new business. Path: User → interacted with → Introduction to Business: 10 lessons to take you into the business world → shares concept → Net assets → concept of → Start a new business.

✨ LLMKGER (LLM) :
   Considering your background in "Introduction to Business" and "Starting a business", I recommend "Starting a New Enterprise" to further explore business administration. This course, taught by Professor Jian Gao, aligns with your interests. It's a great next step, building on your knowledge from "Green Innovation Theory and Practice" and "Innovation and Entrepreneurship Practice in the Intelligent Era".


🎯 COURS : software engineering

📋 E-KGAT (template) :
   Recommended: software engineering. Path: User → interacted with → software testing → shares concept → System class → concept of → software engineering.

✨ LLMKGER (LLM) :
   You've completed software testing, softwar

In [2]:
import json, os

out_dir = r'D:\Memoire\LLMKGER\explanations'
with open(os.path.join(out_dir, 'comparison_ekgat_vs_llmkger.json'), 'r', encoding='utf-8') as f:
    comparison = json.load(f)
with open(os.path.join(out_dir, 'explanations_llmkger.json'), 'r', encoding='utf-8') as f:
    expl_llm = json.load(f)

# Créer un index par user pour croiser les infos
llm_by_user = {e['user']: e for e in expl_llm}

# Critères de sélection : exemples VARIÉS et RICHES
# 1. Continuité de domaine (historique cohérent avec la reco)
# 2. Changement de domaine (nouvelle direction)
# 3. Avec prérequis manquants
# 4. Avec prof réputé

print("=== Sélection d'exemples variés pour la thèse ===\n")

selected = []
for c in comparison:
    e = llm_by_user.get(c['user'], {})
    has_history = len(e.get('user_history', [])) >= 2
    has_concepts = len(e.get('concepts', [])) >= 2
    has_prereq = len(e.get('missing_prereqs', [])) >= 1
    # Score de richesse
    richness = len(e.get('user_history', [])) + len(e.get('concepts', [])) + len(e.get('missing_prereqs', []))
    selected.append((richness, c, e))

# Trier par richesse et prendre les 10 meilleurs
selected.sort(key=lambda x: -x[0])
top10 = selected[:10]

print("Top 10 exemples les plus riches :\n")
for i, (rich, c, e) in enumerate(top10, 1):
    print(f"{i}. {c['course_name']} (richesse={rich})")
    print(f"   User : {c['user']}, historique : {len(e.get('user_history',[]))} cours")

# Sauvegarder les meilleurs exemples
best_examples = [{'user':c['user'], 'course_name':c['course_name'],
                  'ekgat':c['ekgat_explanation'], 'llmkger':c['llmkger_explanation'],
                  'history':e.get('user_history',[]), 'concepts':e.get('concepts',[]),
                  'missing_prereqs':e.get('missing_prereqs',[])}
                 for rich,c,e in top10]
with open(os.path.join(out_dir, 'best_examples_thesis.json'), 'w', encoding='utf-8') as f:
    json.dump(best_examples, f, ensure_ascii=False, indent=2)
print(f"\n💾 10 meilleurs exemples sauvegardés : best_examples_thesis.json")

=== Sélection d'exemples variés pour la thèse ===

Top 10 exemples les plus riches :

1. Start a new business (richesse=14)
   User : U_100090, historique : 5 cours
2. software engineering (richesse=14)
   User : U_1000902, historique : 5 cours
3. Basic Human Biology: Cells and Tissues (richesse=14)
   User : U_1002389, historique : 5 cours
4. English Writing Guidance—Introduction to Writing (richesse=14)
   User : U_100281, historique : 5 cours
5. British and American Music and Culture (richesse=14)
   User : U_10032068, historique : 5 cours
6. computer network (richesse=14)
   User : U_1003625, historique : 5 cours
7. Probability Theory and Mathematical Statistics (richesse=14)
   User : U_10036806, historique : 5 cours
8. College Physics 2 (Electromagnetism, Optics and Quantum Physics) (richesse=14)
   User : U_1006407, historique : 5 cours
9. photojournalism (richesse=14)
   User : U_10135999, historique : 5 cours
10. Marketing in life (richesse=14)
   User : U_10136534, historique

In [3]:
import json, os, re
import numpy as np
from collections import Counter

out_dir = r'D:\Memoire\LLMKGER\explanations'
with open(os.path.join(out_dir, 'explanations_llmkger.json'), 'r', encoding='utf-8') as f:
    expl_llm = json.load(f)

print("=== VOLET 1 : Métriques automatiques ===\n")

# ---------- 1. FAITHFULNESS ----------
# % de faits (historique, concepts) réellement mentionnés dans l'explication
def compute_faithfulness(e):
    text = e['explanation'].lower()
    facts_to_check = []
    # Cours de l'historique
    for h in e.get('user_history', []):
        facts_to_check.append(h.lower())
    # Concepts du cours
    for c in e.get('concepts', []):
        facts_to_check.append(c.lower())
    if not facts_to_check:
        return None
    # Combien de faits sont mentionnés (correspondance partielle : mots-clés)
    mentioned = 0
    for fact in facts_to_check:
        # extraire les mots significatifs du fait (>3 lettres)
        words = [w for w in re.findall(r'\w+', fact) if len(w) > 3]
        if words and any(w in text for w in words):
            mentioned += 1
    return mentioned / len(facts_to_check)

faith_scores = [compute_faithfulness(e) for e in expl_llm]
faith_scores = [s for s in faith_scores if s is not None]
print(f"1. FAITHFULNESS")
print(f"   Score moyen : {np.mean(faith_scores):.3f}")
print(f"   (% des faits KG réellement mentionnés dans l'explication)")

# ---------- 2. READABILITY ----------
# Longueur + score de lisibilité Flesch approximatif
def count_syllables(word):
    word = word.lower()
    vowels = "aeiouy"
    count = sum(1 for i,c in enumerate(word) if c in vowels and (i==0 or word[i-1] not in vowels))
    return max(1, count)

def flesch_score(text):
    sentences = max(1, len(re.split(r'[.!?]+', text)) - 1)
    words = re.findall(r'\w+', text)
    if not words: return 0
    syllables = sum(count_syllables(w) for w in words)
    return 206.835 - 1.015*(len(words)/sentences) - 84.6*(syllables/len(words))

lengths = [len(re.findall(r'\w+', e['explanation'])) for e in expl_llm]
flesch = [flesch_score(e['explanation']) for e in expl_llm]
print(f"\n2. READABILITY")
print(f"   Longueur moyenne : {np.mean(lengths):.1f} mots")
print(f"   Score Flesch moyen : {np.mean(flesch):.1f} (60-70 = facile à lire)")

# ---------- 3. CONSISTENCY (diversité lexicale) ----------
# Les explications sont-elles variées (pas des copiés-collés) ?
all_texts = [e['explanation'] for e in expl_llm]
unique_ratio = len(set(all_texts)) / len(all_texts)
# Diversité des débuts de phrase
starts = [t[:20] for t in all_texts]
start_diversity = len(set(starts)) / len(starts)
print(f"\n3. CONSISTENCY / DIVERSITÉ")
print(f"   Explications uniques : {unique_ratio:.1%}")
print(f"   Diversité des débuts : {start_diversity:.1%}")
print(f"   (élevé = pas de copié-collé, chaque explication est adaptée)")

# Sauvegarder
auto_metrics = {
    'faithfulness': float(np.mean(faith_scores)),
    'readability': {'avg_length': float(np.mean(lengths)), 'flesch': float(np.mean(flesch))},
    'consistency': {'unique_ratio': unique_ratio, 'start_diversity': start_diversity},
}
with open(os.path.join(out_dir, 'L8_auto_metrics.json'), 'w') as f:
    json.dump(auto_metrics, f, indent=2)
print(f"\n💾 Métriques automatiques sauvegardées : L8_auto_metrics.json")

=== VOLET 1 : Métriques automatiques ===

1. FAITHFULNESS
   Score moyen : 0.450
   (% des faits KG réellement mentionnés dans l'explication)

2. READABILITY
   Longueur moyenne : 54.1 mots
   Score Flesch moyen : 30.4 (60-70 = facile à lire)

3. CONSISTENCY / DIVERSITÉ
   Explications uniques : 100.0%
   Diversité des débuts : 34.0%
   (élevé = pas de copié-collé, chaque explication est adaptée)

💾 Métriques automatiques sauvegardées : L8_auto_metrics.json


In [4]:
import json, os, re
import numpy as np

out_dir = r'D:\Memoire\LLMKGER\explanations'
with open(os.path.join(out_dir, 'comparison_ekgat_vs_llmkger.json'), 'r', encoding='utf-8') as f:
    comparison = json.load(f)
with open(os.path.join(out_dir, 'explanations_llmkger.json'), 'r', encoding='utf-8') as f:
    expl_llm = json.load(f)

# Reconstruire les faits par user pour E-KGAT aussi
llm_by_user = {e['user']: e for e in expl_llm}

def count_syllables(word):
    word = word.lower(); vowels = "aeiouy"
    return max(1, sum(1 for i,c in enumerate(word) if c in vowels and (i==0 or word[i-1] not in vowels)))

def flesch_score(text):
    sentences = max(1, len(re.split(r'[.!?]+', text)) - 1)
    words = re.findall(r'\w+', text)
    if not words: return 0
    return 206.835 - 1.015*(len(words)/sentences) - 84.6*(sum(count_syllables(w) for w in words)/len(words))

def faithfulness(text, facts):
    text = text.lower()
    if not facts: return None
    mentioned = 0
    for fact in facts:
        words = [w for w in re.findall(r'\w+', fact.lower()) if len(w) > 3]
        if words and any(w in text for w in words):
            mentioned += 1
    return mentioned / len(facts)

# Calculer pour les deux systèmes
metrics = {'E-KGAT': {'faith':[], 'len':[], 'flesch':[]},
           'LLMKGER': {'faith':[], 'len':[], 'flesch':[]}}

for c in comparison:
    e = llm_by_user.get(c['user'], {})
    facts = e.get('user_history', []) + e.get('concepts', [])
    for system, key in [('E-KGAT','ekgat_explanation'), ('LLMKGER','llmkger_explanation')]:
        txt = c[key]
        f = faithfulness(txt, facts)
        if f is not None: metrics[system]['faith'].append(f)
        metrics[system]['len'].append(len(re.findall(r'\w+', txt)))
        metrics[system]['flesch'].append(flesch_score(txt))

# Diversité
ekgat_texts = [c['ekgat_explanation'] for c in comparison]
llm_texts = [c['llmkger_explanation'] for c in comparison]

print("="*60)
print("  COMPARAISON L8 : E-KGAT vs LLMKGER")
print("="*60)
print(f"{'Métrique':<20}{'E-KGAT':>12}{'LLMKGER':>12}")
print("-"*60)
print(f"{'Faithfulness':<20}{np.mean(metrics['E-KGAT']['faith']):>12.3f}{np.mean(metrics['LLMKGER']['faith']):>12.3f}")
print(f"{'Longueur (mots)':<20}{np.mean(metrics['E-KGAT']['len']):>12.1f}{np.mean(metrics['LLMKGER']['len']):>12.1f}")
print(f"{'Readability (Flesch)':<20}{np.mean(metrics['E-KGAT']['flesch']):>12.1f}{np.mean(metrics['LLMKGER']['flesch']):>12.1f}")
print(f"{'Uniques':<20}{len(set(ekgat_texts))/len(ekgat_texts):>12.1%}{len(set(llm_texts))/len(llm_texts):>12.1%}")
print("="*60)

  COMPARAISON L8 : E-KGAT vs LLMKGER
Métrique                  E-KGAT     LLMKGER
------------------------------------------------------------
Faithfulness               0.338       0.450
Longueur (mots)             25.8        54.1
Readability (Flesch)         2.0        30.4
Uniques                   100.0%      100.0%


In [7]:
from groq import Groq
import json, re

# Remets ta clé Groq
GROQ_API_KEY = "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"
client = Groq(api_key=GROQ_API_KEY)

# Tester sur UN exemple
c = comparison[0]
prompt = f"""You are evaluating the quality of a course recommendation explanation. Rate it on 4 criteria from 1 (poor) to 5 (excellent).

COURSE: {c['course_name']}
EXPLANATION: "{c['llmkger_explanation']}"

Rate these criteria (respond ONLY with a JSON object, no other text):
- faithfulness: does it reference concrete facts (courses, concepts, instructor)?
- helpfulness: does it help the learner understand why this course fits them?
- readability: is it clear and easy to read?
- consistency: is it coherent and well-structured?

Respond exactly in this format:
{{"faithfulness": X, "helpfulness": X, "readability": X, "consistency": X}}"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role":"user","content":prompt}],
    temperature=0.3, max_tokens=150,
)
raw = response.choices[0].message.content
print("=== RÉPONSE BRUTE ===")
print(repr(raw))
print("\n=== AFFICHÉE ===")
print(raw)

m = re.search(r'\{[^}]+\}', raw)
if m:
    print("\n=== JSON EXTRAIT ===", m.group())
    try:
        print("Parsing OK :", json.loads(m.group()))
    except Exception as e:
        print("Erreur :", e)

=== RÉPONSE BRUTE ===
'{"faithfulness": 5, "helpfulness": 4, "readability": 5, "consistency": 5}'

=== AFFICHÉE ===
{"faithfulness": 5, "helpfulness": 4, "readability": 5, "consistency": 5}

=== JSON EXTRAIT === {"faithfulness": 5, "helpfulness": 4, "readability": 5, "consistency": 5}
Parsing OK : {'faithfulness': 5, 'helpfulness': 4, 'readability': 5, 'consistency': 5}


In [8]:
import json, os, time, re
import numpy as np

out_dir = r'D:\Memoire\LLMKGER\explanations'
with open(os.path.join(out_dir, 'comparison_ekgat_vs_llmkger.json'), 'r', encoding='utf-8') as f:
    comparison = json.load(f)

def judge_explanation(course_name, explanation):
    prompt = f"""You are evaluating the quality of a course recommendation explanation. Rate it on 4 criteria from 1 (poor) to 5 (excellent).

COURSE: {course_name}
EXPLANATION: "{explanation}"

Rate these criteria (respond ONLY with a JSON object, no other text):
- faithfulness: does it reference concrete facts (courses, concepts, instructor)?
- helpfulness: does it help the learner understand why this course fits them?
- readability: is it clear and easy to read?
- consistency: is it coherent and well-structured?

Respond exactly in this format:
{{"faithfulness": X, "helpfulness": X, "readability": X, "consistency": X}}"""
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role":"user","content":prompt}],
            temperature=0.3, max_tokens=100,
        )
        txt = response.choices[0].message.content.strip()
        m = re.search(r'\{[^}]+\}', txt)
        if m:
            return json.loads(m.group())
    except Exception as e:
        time.sleep(2)
    return None

np.random.seed(42)
sample_idx = np.random.choice(len(comparison), 30, replace=False)

scores = {'E-KGAT': [], 'LLMKGER': []}
print("Évaluation LLM-juge (30 × 2)...\n")

for count, i in enumerate(sample_idx):
    c = comparison[i]
    for system, key in [('E-KGAT','ekgat_explanation'), ('LLMKGER','llmkger_explanation')]:
        judgment = judge_explanation(c['course_name'], c[key])
        if judgment:
            scores[system].append(judgment)
        time.sleep(0.5)
    if (count+1) % 10 == 0:
        print(f"  {count+1}/30...")

print("\n" + "="*60)
print("  VOLET 2 : LLM-JUGE (échelle 1-5)")
print("="*60)
print(f"{'Critère':<16}{'E-KGAT':>12}{'LLMKGER':>12}")
print("-"*60)
for crit in ['faithfulness','helpfulness','readability','consistency']:
    e_scores = [s[crit] for s in scores['E-KGAT'] if crit in s]
    l_scores = [s[crit] for s in scores['LLMKGER'] if crit in s]
    print(f"{crit:<16}{np.mean(e_scores):>12.2f}{np.mean(l_scores):>12.2f}")
print("="*60)
print(f"\nÉvaluations réussies : E-KGAT {len(scores['E-KGAT'])}/30, LLMKGER {len(scores['LLMKGER'])}/30")

with open(os.path.join(out_dir, 'L8_llm_judge.json'), 'w') as f:
    json.dump({'E-KGAT':scores['E-KGAT'], 'LLMKGER':scores['LLMKGER']}, f, indent=2)
print("💾 Sauvegardé : L8_llm_judge.json")

Évaluation LLM-juge (30 × 2)...

  10/30...
  20/30...
  30/30...

  VOLET 2 : LLM-JUGE (échelle 1-5)
Critère               E-KGAT     LLMKGER
------------------------------------------------------------
faithfulness            3.40        4.83
helpfulness             2.13        3.97
readability             2.40        4.93
consistency             2.67        4.80

Évaluations réussies : E-KGAT 30/30, LLMKGER 30/30
💾 Sauvegardé : L8_llm_judge.json


In [9]:
import json, os
import numpy as np

out_dir = r'D:\Memoire\LLMKGER\explanations'

# Charger les résultats des 2 volets
with open(os.path.join(out_dir, 'L8_auto_metrics.json')) as f:
    auto = json.load(f)
with open(os.path.join(out_dir, 'L8_llm_judge.json')) as f:
    judge = json.load(f)

# Calculer les moyennes du LLM-juge
def avg_crit(data, crit):
    return float(np.mean([s[crit] for s in data if crit in s]))

l8_summary = {
    'volet_1_automatique': {
        'LLMKGER': {
            'faithfulness': auto['faithfulness'],
            'readability_flesch': auto['readability']['flesch'],
            'avg_length': auto['readability']['avg_length'],
            'uniqueness': auto['consistency']['unique_ratio'],
        }
    },
    'volet_2_llm_juge': {
        'E-KGAT': {c: avg_crit(judge['E-KGAT'], c) for c in ['faithfulness','helpfulness','readability','consistency']},
        'LLMKGER': {c: avg_crit(judge['LLMKGER'], c) for c in ['faithfulness','helpfulness','readability','consistency']},
    },
    'conclusion': 'LLMKGER surpasse E-KGAT sur tous les critères (automatique + LLM-juge concordent)'
}

with open(os.path.join(out_dir, 'L8_summary_final.json'), 'w') as f:
    json.dump(l8_summary, f, indent=2, ensure_ascii=False)

print("=== RÉCAPITULATIF L8 ===\n")
print("VOLET 1 (automatique) — LLMKGER vs E-KGAT :")
print(f"  Faithfulness : 0.450 vs 0.338")
print(f"  Readability (Flesch) : 30.4 vs 2.0")
print(f"  Longueur : 54 vs 26 mots")
print(f"\nVOLET 2 (LLM-juge, 1-5) — E-KGAT vs LLMKGER :")
for c in ['faithfulness','helpfulness','readability','consistency']:
    print(f"  {c.capitalize()} : {avg_crit(judge['E-KGAT'],c):.2f} vs {avg_crit(judge['LLMKGER'],c):.2f}")
print(f"\n💾 Sauvegardé : L8_summary_final.json")

=== RÉCAPITULATIF L8 ===

VOLET 1 (automatique) — LLMKGER vs E-KGAT :
  Faithfulness : 0.450 vs 0.338
  Readability (Flesch) : 30.4 vs 2.0
  Longueur : 54 vs 26 mots

VOLET 2 (LLM-juge, 1-5) — E-KGAT vs LLMKGER :
  Faithfulness : 3.40 vs 4.83
  Helpfulness : 2.13 vs 3.97
  Readability : 2.40 vs 4.93
  Consistency : 2.67 vs 4.80

💾 Sauvegardé : L8_summary_final.json
